# Uniaxial compression, TWO-PISTON — single strain level

Analysis of **one** applied-strain level of a `uniaxial_compression` run: drained consolidation at
**constant bath pressure** — both solvent reservoirs are closed by NPT-pistons (Marioni et al., J. Membr. Sci.
738 (2026) 124837, Eq. 3) held at `P_target`, and a solvent-transparent **load piston** loads the network through
the same cumulative strain sweep as `uniaxial_compression`.  Set `LEVEL` and the run identifiers in **Config**,
run the **sync** cell once, then run everything.  All analysis code lives in `scripts/lib/uniaxial.py`; this
notebook only configures it and draws the eleven one-piston figures **plus two two-piston panels**: the wet-piston
bath check (P_feed, P_perm measured vs `P_target` over the hold) and the solvent expelled (wet-piston
displacement).  The load piston plays the role of "the piston" everywhere (its force is the network load; the
piston files carry it in the first value column, `[load | feed | perm]`).

Method notes and the file list are in **Notes** at the end (2026-09-16).

## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: uniaxial.py (all analysis code) + volfrac.py
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import uniaxial as uni
uni = importlib.reload(uni)            # pick up edits to lib/uniaxial.py without a kernel restart
uni.setup_style()
print('analysis code: ', LIB / 'uniaxial.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
LEVEL = "0.10"      # the ONE applied-strain level analysed here (one of STRAIN_TARGETS in the .batch; files _c<LEVEL>)
cfg = uni.Config(
    DATANAME    = "final_config_slab_support_periodic_10beads_wide_rho04_random_1.0_1.0_16000000_two_pist_pm40",  # RANDOM f=4 periodic slab (chain_random_periodic.batch, 2026-11-02)
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = None,        # <steps> tag = the flat HOLD LENGTH (HOLD_AUTO=0 in the _random.batch); None resolves it from the files
    RUN_ID      = "random_16M_comp_1",  # local folder under flow_data_local/{raw_data,plots}/uniaxial_compression
                                  # job 55793779 (2026-11-17, SEAT_MODE=rg, UNLOAD_MARGIN=8): complete, 13.65 M steps -- hold 4 M (_c0.10)
                                  # and the full unload (_u0.10).  See runs/README.md
    COMP_LEVELS = [LEVEL],
    mode        = "compression",  # two-piston COMPRESSION run ...
    # ---- measurement windows ----
    plateau_frac      = 0.25,
    plateau_frac_auto = 0.45,
    # ---- figure conventions (2026-10-06) ----
    PARTIAL_NORM = 'share',       # 'share': under the total-stress, partial-stress and thermodynamic-pressure figures, a second figure
                                  # with the back pressure removed and divided by the driving pressure (see Notes); 'raw': originals only
    FLIP_Z      = True,           # z-profiles drawn with the load piston on the LEFT (1 − z/L, 0 → 1), support on the right (plotting only)
    G_REF       = None,           # shear modulus guide ΔP_th = (4/3)(G/M) ΔP on the normalised P_th figure.  0.2 was the LATTICE gel's;
                                  # for this random gel put the simple_shear_single value of RUN_ID two_pist_rho04_iso_random_8M_9M_shear_7
    G_COMP_REF  = None,           # compression-mode G guide: 0.04-0.06 was the lattice gel's; this run reads its own from fig_G (~0.39)
    # ---- solvent volume fractions (lib/volfrac.py) ----
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, P_CAL = 1.5,
    # ---- M and G as increments from the eps = 0 reference ----
    M_SUBTRACT_REF = True, G_SUBTRACT_REF = True,
    # ---- D_c fit knobs (modes, face trim) ----
    DC_FREE_AMPS = True, DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_SLOW_REF = 0.17, DC_TARGET_RESID = 0.01,
    DC_PRIMARY  = 'unload',       # the D_c kappa and the summary use: the free re-swelling profile fit (the hold fits are archived, 2026-11-17)
    # ---- a sweep's small-strain M (third estimate in fig 7), drawn only when the sweep is of the same network as this run ----
    M_RECORD    = "../../flow_data_local/raw_data/uniaxial_compression/periodic_rho04_14M_9M_twoPlatesMove_7_sweep/M_record.json",   # the diamond-LATTICE gel's sweep (run 7): not drawn for this random gel; point it at the random gel's sweep once that exists
)
# Every other knob keeps its default -- see `uni.Config` in lib/uniaxial.py.  P_BARO (= P_target of the
# wet pistons, 1.5) normalises the total-stress panels and is the reference line of the bath check.

In [ ]:
# Pull the files this notebook reads from the cluster (one login per cluster asked).
#   CLUSTER in the Config picks the source: 'auto' (default: the cluster this run was last pulled from, else Expanse, then Pod
#   when the campus VPN is up), 'expanse' (password + TOTP) or 'pod' (password; needs the VPN).
# the file list includes piston_pressure / permeation / pressure_reservoirs (two-piston runs, the Config default).
SYNC, FORCE_SYNC = True, False
if SYNC:
    uni.sync_from_expanse(cfg, levels=[LEVEL], force=FORCE_SYNC)

In [ ]:
# Load + compute everything:
#   R  -- the eps = 0 reference state (also the zero-strain bath pressures on the wet pistons)
#   L  -- this level: stresses, Terzaghi split, load-piston plateau, M, G, the unload D_c and kappa, + L['wet'] (bath check)
R = uni.load_reference(cfg)
L = uni.load_level(cfg, R, LEVEL)
assert L is not None, f'level {LEVEL}: core files missing -- run the sync cell'
uni.add_volume_fractions(cfg, R, [L])
uni.print_summary(cfg, [L])
uni.save_M_record(cfg, R, [L], path=cfg.DATA_DIR / 'M_record.json')   # this run's own one-level record: the permeation notebook's M_RECORD;
                                                                      # with an unload D_c it also writes the record the permeation BATCH plans dP / NSTEPS from (lib/perm_plan.py; lsync to send it)


## 2 · Figures

In [ ]:
# 1 · Strain diagnostic -- solid ε_Rg, dashed ε_BB, faint ε_piston (load piston vs support); shaded = plateau window.
#     Runs WITH an unload (deck Phase 2c, files _u<LEVEL>; the _fine / _andrelax batches): panel (a) = ramp + hold,
#     panel (b) = retraction to the seated gap + free re-swelling (strains fall from the level towards 0 = reference).
uni.fig_strain(cfg, R, [L]);


In [ ]:
# 1b · Steady displacement profile u_z(Z) of the polymer -- plateau frames vs the ε = 0 reference, per atom, binned by the
#      reference position: u_z = u_0 − ε_disp (Z − Z^P) under uniform network stress, so the fitted slope is the strain of the
#      network alone (the applied strain also counts the gel's rigid fall onto the support).  ε_disp divides M, G and κ
#      (M_STRAIN = 'disp', the default since 2026-10-03; 'rg' | 'applied' to compare).  (b) residuals from the line.
uni.fig_disp_profile(cfg, R, L);

In [ ]:
# 1c · Crosslinker fluctuations of the ε = 0 reference (traj_ref only; bond topology rebuilt from the frames, lib/xlink.py).
#      (a) Γ = ⟨ΔR²⟩/⟨R̄²⟩ per strand, (b) extension x = √⟨R²⟩/(n b); the symbols are defined in the table under the panels.
uni.fig_xlink(cfg, R);

In [ ]:
# 2 · Solvent volume fraction φ_s: mass fraction, Voronoi and λ-calibrated Voronoi -- reference (ε = 0) vs compressed
uni.fig_volfrac(cfg, R, L);

In [ ]:
# 3 · Total stress evolution σ^t_zz, σ^t_xx, σ^t_yy / P_bath -- reference (dashed) -> hold (cividis) -> plateau (bold).
#     Pore baseline = the FEED reservoir interior (the box top is vacuum in the two-piston geometry).
#     Underneath (PARTIAL_NORM = 'share'): the same panels with the back pressure removed, (σ^t − P_bath)/ΔP_pist --
#     zz reads 1 in the gel and 0 in both reservoirs once the hold has relaxed.
#     (b) UNLOAD companion (runs with _u<LEVEL> files): the same figures over the free re-swelling, σ_zz only (the deck
#     records no lateral stress in Phase 2c); the normalised one uses the hold's ΔP_pist, so it relaxes 1 -> 0.
uni.fig_total_stress(cfg, R, L);


In [ ]:
# 4 · Solvent and polymer partial σ_zz evolutions with the total superimposed.
#     Underneath (PARTIAL_NORM = 'share'): solvent*, polymer*, total* -- each species' share of the back pressure removed
#     and divided by ΔP_pist:  w_s = σ_s/σ^t per bin and snapshot,  σ_s* = (σ_s − w_s P_bath)/ΔP_pist,
#     σ_p* = (σ_p − (1 − w_s) P_bath)/ΔP_pist,  σ^t* = (σ^t − P_bath)/ΔP_pist.  See Notes.
#     (b) UNLOAD companion when the level has _u<LEVEL> files: compressed -> re-swollen (reference dashed).
uni.fig_partial_stress(cfg, R, L);


In [ ]:
# 5 · Network stress σ'_zz, σ'_xx, σ'_yy (Terzaghi: σ' = σ^t − p_pore): reference (ε = 0, dashed, ~0) and FINAL
#     equilibrated state only, 95 % bands.  No evolution curves: p_pore is not uniform while the gel consolidates.
#     (b) UNLOAD companion when the level has _u<LEVEL> files: σ'_zz after the free re-swelling over the reference and
#     the end-of-hold plateau profile -- does the network stress return to the reference?  (zz only)
uni.fig_network_stress(cfg, R, L);


In [ ]:
# 5b · Reservoir normal-stress CHECK (2026-10-06): σ^t_xx, σ^t_yy, σ^t_zz over the hold in the gel interior (solid) and in the
#      two solvent reservoirs (feed dashed, permeate dotted) + plateau profiles.  The reservoirs are the in-situ control: a fluid
#      must read P_bath in all three components through the whole hold.  A lateral rise confined to the gel is the network's
#      σ'_xx (G_comp/M = ½(1 − σ'_xx/σ'_zz), quoted under the figure); a lateral drift in the reservoirs would be a box / normalisation artefact.
uni.fig_reservoir_normal_stress(cfg, R, L);


In [ ]:
# 6 · LOAD-piston pressure P = F_z/A vs step, linear + log; green = auto-selected plateau window.
#     (b) UNLOAD companion when the level has _u<LEVEL> files: the load over the retraction + re-swelling (block means
#     only -- the deck writes no raw piston_force in Phase 2c); dashed = plates stopped at the seated gap.
uni.fig_piston(cfg, R, L);


In [ ]:
# 7 · Longitudinal modulus M (increments from ε = 0): network vs load piston, both divided by ε_disp (the steady
#     displacement-profile strain, M_STRAIN = 'disp').  A sweep's small-strain stress-strain slope (M_RECORD) is added as a
#     third estimate only when that sweep was run on the same network (diamond | rot45 | random); none exists for the random gel yet.
uni.fig_M(cfg, R, L);


In [ ]:
# 8 · Network-stress anisotropy σ'_zz/σ'_xx and σ'_zz/σ'_yy vs step  (= M/(M − 2G))
uni.fig_ratio(cfg, R, L);

In [ ]:
# 9 · Shear modulus G = (σ'_zz − σ'_ii)/(2ε) from xx and from yy
uni.fig_G(cfg, R, L);

In [ ]:
# 10 · Cooperative diffusivity D_c from the UNLOAD: the free re-swelling profiles after the hold (deck Phase 2c, files
#      _u<LEVEL>) and their modal fit, tau_1 = L^2/(pi^2 D_c).  This is the D_c that kappa and the cross-deck comparison use.
uni.fig_Dc_reswelling(cfg, R, L);


In [ ]:
# 10b · The profile fit of figure 10 against TIME: each snapshot's profile projected on the fit's cosine modes gives one amplitude
#       per mode; drawn as the first mode's share of the thickness change (dots) against the fitted decay (line), residuals underneath.
#       Same data and same fit as figure 10 -- a second view of it, not a second estimate.  Notes in the cell below.
uni.fig_Dc_reswelling_time(cfg, R, L);

**Figures 10 and 10b: one fit, two views** (this cell is notebook text only; it is not part of the saved PNGs)

*The model.*  The gel is treated as a 1-D poroelastic slab: the network strain obeys a diffusion equation,
$\partial_t \varepsilon = D_c\,\partial_z^2 \varepsilon$, with $D_c = \kappa M$.  At the end of the hold the strain is uniform.  Once the plates
have left, both faces are free (zero network stress), the strain at the faces takes its final value and the interior follows by diffusion
from both sides.  Written for the displacement $u_z$ of each bin since the plates stopped, with $\zeta\in[0,1]$ the material coordinate
across the compressed gel of thickness $L$:

$$\frac{u_z(\zeta,t)}{L} = \sum_{k=1}^{N} B_k\cos(k\pi\zeta)\left[e^{-k^2\pi^2 D_c t/L^2}-1\right],\qquad \tau_k=\frac{L^2}{k^2\pi^2 D_c}.$$

All bins of all snapshots are fitted at once.  $D_c$ is the one shared number that sets every mode's decay time; the amplitudes $B_k$ are
free, so the fit does not need to know how far the gel ends up re-swelling.

*Figure 10* draws the fitted data as it is: displacement profiles across the gel, one curve per snapshot, coloured by time.

*Figure 10b* draws the same data and the same fit against time.  The fitted data is a surface $u_z(\zeta,t)$, not one curve against
time, so to get a time series each snapshot's profile is projected on the cosine modes, which gives one amplitude $a_k(t)$ per mode.
The gel's thickness change is $u_z(1)-u_z(0) = -2L\sum_{k\ \mathrm{odd}} a_k(t)$ (even modes are symmetric and do not change the thickness),
so the panel draws $-2L\,a_1(t)$ for the first mode (dots) against the fitted $-2L\,B_1\,[e^{-t/\tau_1}-1]$ (line).  The first mode
carries nearly all of the thickness change here; the higher odd modes are at the noise level (their fitted asymptotes are in the
figure title), because they decay 9 and 25 times faster and are largely gone by the time the plates have stopped.  Nothing new is
fitted: 10b is the same $D_c$ as figure 10, shown as timing instead of shape, not an independent estimate.

*Why there is no thickness-trace $D_c$* (removed 2026-11-21).  The gel thickness against time can be fitted directly with
$\Delta L(t) = \Delta L_\infty\,[1-\sum_{k\ \mathrm{odd}} \tfrac{8}{k^2\pi^2}e^{-t/\tau_k}]$, using either the bounding-box thickness or the Rg
thickness.  These fits are no longer made or shown, for three reasons.

* The **bounding-box thickness** is set by the two outermost beads.  The loose surface fringe springs out within ~100 τ of the retraction,
  which is not diffusion through the slab; on `random_16M_comp_1` no version of the fit described it ($R^2$ 0.70 at best).
* With the **asymptote fixed** at $\Delta L_\infty = L_\mathrm{ref}-L_\mathrm{hold}$, the fit assumes the gel returns exactly to its seated
  reference.  On the rg-seated runs it does not: the Rg thickness of `random_16M_comp_1` goes from 68.6 at the end of the hold to 79.5,
  past the seated reference of 74.4, and the fit failed ($R^2$ −2.9, $D_c$ 0.44).
* With the **asymptote free**, the Rg trace did fit ($D_c$ 0.105, $R^2$ 0.985, against the profile fit's 0.123), but it is the same
  re-swelling event reduced to one number per time step.  It cannot separate the modes, its asymptote and $D_c$ trade off against each
  other, and its clock starts while the plates are still retracting.  It is a weaker fit of the same data, not an independent measurement.

In [ ]:
# 11 · κ = D_c/M (hydraulic permeability / viscosity) from the network and the piston M, with the UNLOAD D_c (DC_PRIMARY = 'unload').
uni.fig_kappa(cfg, R, L);


In [ ]:
# 12 · TWO-PISTON: bath check -- P_feed and P_perm measured on the wet pistons vs P_target over the hold (+ P_load).
#      (b) UNLOAD companion when the level has _u<LEVEL> files: the same check over the retraction + re-swelling.
uni.fig_wet_pistons(cfg, R, L);


In [ ]:
# 13 · Solvent exchanged with the gel = change of the gel volume, A·ΔL (incompressible polymer, laterally fixed box):
#      solvent EXPELLED over the drive + hold, from the bounding-box thickness (the plate gap under load) and the Rg (bulk) thickness;
#      circles = direct count of the solvent between the plates (solvent_density_z snapshots), a check on the bounding-box curve.
#      (b) UNLOAD companion when the level has _u<LEVEL> files: solvent ABSORBED over the retraction + re-swelling (no density file
#      is written in the unload, so no direct count; the bounding box follows the surface fringe, the Rg curve is the bulk uptake).
uni.fig_solvent_expelled(cfg, R, L);


In [ ]:
# 14 · Thermodynamic pressure P_th = −⅓ tr(σ^t) evolution (positive under compression; dotted = P_bath).
#      Underneath (PARTIAL_NORM = 'share'): the solvent, polymer and total traces with the back pressure removed, / ΔP_pist
#      (shares from the traces, w_s = tr σ_s / tr σ^t); the total is the raw P_th shifted and rescaled.  Dash-dot guide:
#      1 − (4/3)(G/M), the trace of the load the lateral stresses imply (G = G_REF, M = M_net)
uni.fig_thermo_pressure(cfg, R, L);

In [ ]:
# 15 · Osmotic pressure Π = −⅓ tr(σ′): reference (ε = 0) and final equilibrated state, 95 % bands
uni.fig_osmotic_pressure(cfg, R, L);


## Notes

Everything below is reference material — nothing above depends on reading it.

### D_c: unload (trusted) vs load (hold) — 2026-10-21

**2026-11-17: the load (hold) D_c fits are archived.**  `lib/uniaxial.py` no longer fits the hold: the compression D_c is the
unload (free re-swelling) profile fit, κ divides that D_c only, and a level without an unload has no D_c or κ.  The hold fits,
the D_c(t) and D_c(φ_s) fits, their figures and the hold-adequacy check are in `archive/scripts/lib/uniaxial_load_Dc.py`
(`ldc.add_load_Dc(cfg, R, L)`, `ldc.fig_Dc(cfg, R, L)`).  What follows on the load values is history.

* Every D_c this notebook prints or draws is tagged by where it comes from.  **UNLOAD** = the free re-swelling after the hold
  (files `_u<LEVEL>`): the profile fit (figures 10 and 10b; `L['unload']['prof']`) is the only estimate.  **LOAD** = the hold after the compression ramp: the profile fit (`L['Dc']`) and the
  load-piston stress trace (`L['Dc_stress']`).  `print_summary` groups the columns the same way; κ carries a `u`/`h` tag for the
  D_c it divides by.  The cross-deck comparison notebook uses the unload D_c only.
* Why the load values are not trusted (amplitude-spectrum analysis of this run's hold, 2026-10-20).  The ramp at 0.04 σ/τ takes
  ~315 τ against τ₁ ≈ 6000 τ, so it should leave face skins of depth √(D_c t_ramp) ≈ 4 σ and an unstrained interior.  Fitting the
  hold with the mode amplitudes the ramp predicts, A_m ∝ (1 − e^{−λ_m t_r})/m³: (i) the decay rates give 0.051 (the lib fit),
  the spectrum shape 0.065–0.085, a fit with no free amplitudes 0.063–0.067 — the hold D_c depends on what is assumed about the
  onset state; (ii) the mode-1 amplitude, which is D_c-independent, is half the prediction: the hold redistributes 0.058 of
  strain (0.062 from the interior slope change) against a closure of 0.111 and a plateau profile strain of 0.082, so the
  interior was already strained ~0.02 at the hold onset.  The Rg gel extent fell 4.5 σ during the ramp (bounding box 12 σ), the
  load piston spiked to ~0.3 during the ramp against a drained increment of 0.03, and the interior strain change is slightly
  negative for the first ~1500 τ of the hold: an undrained response of the ρ = 0.47 solvent relaxing (the "early lag" of the
  D_c(t) figure).  The 1-D incompressible consolidation model has no such storage term.  The unload starts from a uniform
  state with free faces and shows none of this.

### Why the unload is primary (2026-10-15)

* On the fine-cadence run with an unload (`_andrelax`), the hold's consolidation fit gives D_c = 0.051 and the free re-swelling
  profile fit 0.120, with the stress trace at 0.069 between them.  No test moved the hold value: the finite-difference convention
  test passes for all fitters, excluding the faces from the fit does not raise it, windows starting after the ramp-end skin has
  relaxed still read 0.05, and a global strain-dependent D(ε) fit gives the hold half the unload's D at **equal** local strain.
  The two transients are different processes (consolidation at fixed thickness vs swelling with free faces), and the re-swelling
  one starts from a uniform state with free faces, so it is the cleaner measurement: `DC_PRIMARY = 'unload'` makes it the D_c
  that κ and the summary use (`L['Dc_primary']`; the hold's κ stays in `L['kappa_hold']`).  Figure 10 draws it alone; figure 11 no longer shows the hold's κ.
* The re-swelling thickness-trace fits (bounding box, Rg) were removed on 2026-11-21; the note under figure 10b says why.
* Figure 7's third M (drawn only when the sweep is of the same network as this run) is the sweep's small-strain stress-strain slope with the displacement strain (`M_RECORD`, run 7: 0.42); this
  level's two values are secants Δσ'/ε_disp and ΔP/ε_disp at ε_disp ≈ 0.08 (0.37-0.38).

### Unload '(b)' companions (2026-10-12)

* Runs with deck **Phase 2c** (`UNLOAD=1` in the batch: the `_fine` / `_andrelax` runs) write a second set of files tagged
  `_u<LEVEL>` -- both plates retracted to the seated reference gap at the drive speeds, then a free re-swelling hold.
  `load_level` builds `L['U']` from them (`uni.load_unload_state`, knob `UNLOAD_FIGS`), a level-like dict, and every
  figure above whose quantity the deck records in Phase 2c draws it as a second, **(b)** figure (files `*_unload.png`):
  strain (panel b), total / partial / network σ_zz (zz only: no lateral stress, density or `strain_zz` file is written
  in the unload), the load piston (block means only), the wet pistons and the solvent expelled.  `fig_Dc` has had its own
  unload row (e, f) since 2026-10-09.  Levels without `_u` files draw exactly what they drew before.
* The unload's "final" window is the last `plateau_frac` of the re-swelling; the normalised stress companions divide by the
  **hold's** ΔP_pist, so they read 1 at the start of the unload and relax towards 0.

### Two-piston specifics (2026-09-16)

* **Geometry**: `zlo` = vacuum | permeate piston (type 6) | permeate reservoir | support (type 4) | gel | feed
  reservoir with the **load piston** (type 7) | feed piston (type 5) | vacuum = `zhi`.  `lb/uniaxial.py` reads the
  four wall planes from `traj_ref`; the load piston is `R['z_piston']` (the loading plate), the wet pistons are
  `R['z_feed']`, `R['z_perm']`.
* **Symmetric drive (2026-09-18)**: every strain level closes the plate gap with the load piston driven **down**
  and the support driven **up**, half the closure each (`drive_split = 0.5` in the `.lmp`; `1.0` = the old
  load-piston-only drive).  `support_position_…_c<lvl>.dat` records the support, and the $D_c$ fit and the
  wall lines use its **held** plane (`L['z_supp']`); the applied strain is still the prescribed gap closure
  over $L_0^{BB}$.
* **Pore baseline (revised 2026-09-22)**: the window used for `p_pore` is the **feed reservoir interior**, not
  `z/Lz ≈ 0.95` (that is vacuum here).  A bin is used only if it lies **entirely ≥ `res_wall_margin` (3 σ) from the
  feed-piston plane** and ≥ 2 bins above the gel top.  The 1 σ piston sheet depletes/layers the solvent out to ~3 σ
  and half of its pair virial is booked on the piston atoms, which the solvent profile does not count, so the bin
  touching the piston reads σ_zz ≈ 1.43 for a 1.50 bath; with the old `feed piston − 1.5 σ` bound that bin was in
  the window, `p_pore` came out 0.016 low and **every** σ′ (reference curves, M_network, G, the anisotropy
  ratio) carried a +0.016 offset.  The window is rebuilt **per stress snapshot** from the measured feed-piston
  plane (`piston_position_…_c<lvl>.dat`, column `z_feed`), so a piston that drifts while solvent is expelled
  keeps the same clearance; `R['bw']` is the reference window, `L['bw']` the per-snapshot ones.
* **What moves**: the summary line *plates over the level* reports the start → end plane of the support, the
  load piston and both wet pistons over the level's position files.  Under the symmetric drive the support is
  driven too (+½ of the closure), so no plate is assumed stationary; the fixed box is the frame of every profile.
* **Piston files** carry one column set per piston, `[load | feed | perm]`, with a `# step …` header.  The
  compression loaders read the first value column (load piston = network load), so `M_piston`, `P_ref`, the
  plateau window etc. mean exactly what they do for one-piston runs.  `L['wet']` holds the wet-piston pressures
  (`piston_pressure_…_c<lvl>.dat`) and the solvent expelled (`permeation_…_c<lvl>.dat`, `dV_total`).
* **Bath check**: figure 12 must show `P_feed`, `P_perm` ≈ `P_target` throughout the hold (the pistons regulate
  the reservoirs; a systematic offset means the settle was too short or the damping too strong — see the
  `C_crit` / period printout in the `.lmp` log).  Figure 13 shows where the expelled solvent went; the feed
  piston rise must stay below the converter's `margin_feed`.
* **Files read** (into `flow_data_local/raw_data/uniaxial_compression/<RUN_ID>/`): as `uniaxial_compression_one_pist_single.ipynb` plus
  `piston_pressure[_c<lvl>]`, `permeation_c<lvl>`, `pressure_reservoirs[_ref]`; `piston_force_avg_ref` now has
  three columns.

### Everything else

The one-piston Notes (strain diagnostic, Terzaghi split, plateau window, M, G, D_c, κ, volume fractions) apply
unchanged — see `uniaxial_compression_one_pist_single.ipynb`.

### D_c fit corrected (2026-09-28)

* The hold relaxes a slab **held between two drained plates at the same bath pressure**: both faces pinned
  (`u''(0) = u''(1) = −q/D_c`) and the pore pressure equal at both faces (`ε(0) = ε(1)`, uniform total stress), so
  the strain modes are the *periodic* ones, cos/sin(2mπζ), and `u` relaxes in sin(2mπζ) (antisymmetric, the
  symmetric drive's family) and cos(2mπζ) − 1 (symmetric) at `4m²π²D_c/L²`: **`τ₁ = L²/(4π² D_c)`**.  `fit_Dc`
  used (2ζ − 1) + cos(kπζ), odd k, at `k²π²D_c/L²` — Terzaghi's constant-*load* series bent to vanish at both
  plates — which is not a solution of the held-slab equation: on the exact relaxation with `D_c = 0.12` it returned
  0.60 with R² 0.999.  The decay times were always right; the conversion to `D_c` was ~5× (comp_1 ε = 0.1: 0.49 → 0.043,
  `κ = D_c/M` 1.5 → 0.13, against the permeation `κ = D_c/M ≈ 0.4` and the Darcy `κ = 0.29`).  The hold-adequacy
  line now uses `L²/(4π² D_c)` for the fitted `D_c`; the *slow* line keeps the deck's own sizing formula
  `L²/(π² Dc_est)` (`Dc_est = 0.17` there is 0.043 in this convention: right at ε ≈ 0.1, but the fitted `D_c` falls
  with strain faster than the deck's (1 − ε)² sizing assumes, so from ε ≈ 0.2 up the check reports fewer than 5 τ₁
  held.  Take the high-strain values with care: a hold shorter than ~2 τ₁ constrains `D_c` poorly, and the direct
  measure of what is still unrelaxed is the `RELAX_SYS` tail fit (δ_sys), not this extrapolation).
  `DC_FREE_AMPS = False` (load-control amplitudes) has no meaning for these modes and falls back to free amplitudes.

### Normalised partial stresses and the z orientation (2026-10-06)

* **`PARTIAL_NORM = 'share'`**: under figures 3, 4 and 14 the same call draws a second figure (file stem `…_norm`; the
  originals, their PNGs and the plot numbers are unchanged) with each species' share of the back pressure removed and
  the result divided by the driving pressure — the form of Marioni et al.'s partial `P_zz` figure, whose permeate is at
  ≈ 0 where our bath is at 1.5.  Per z-bin, per snapshot and per component:
  `w_s = σ_s/σ^t`, `σ_s* = (σ_s − w_s P_bath)/ΔP_pist`, `σ_p* = (σ_p − (1 − w_s) P_bath)/ΔP_pist`,
  `σ^t* = (σ^t − P_bath)/ΔP_pist`, with `P_bath = P_BARO` and `ΔP_pist` the plateau load-piston pressure increment of the
  level (`L['dP_pist']`, the one `M_piston` uses).  The shares come from the partial stresses alone — no mass or volume
  fractions, no calibration against another run.  `σ_s* = w_s σ^t*`, `σ_p* = (1 − w_s) σ^t*`, so `σ_s* + σ_p* = σ^t*`
  (asserted in `uni.partial_norm`); in the reservoirs `σ_p = 0`, `w_s = 1`.  Expected and found (comp_1, ε = 0.10,
  `ΔP_pist = 0.032`): total* = 1.00 in the gel interior and 0 in both reservoirs (feed +0.01, permeate −0.05),
  solvent* 0.47, polymer* 0.53; the early-hold curves sit above 1 (up to ≈ 2.7) and relax onto it.  The ε = 0 reference is
  drawn dashed and comes out ≈ 0 (−0.04, bin scatter 0.09): the sanity check.
* **Trace (figure 14)**: the same with `w_s = tr σ_s / tr σ^t`; solvent, polymer and total are all drawn.  Only the
  constant `P_bath` is taken off the total, so the normalised total trace is just the raw `P_th` shifted and rescaled:
  0.84 in the gel, not 1 (the lateral total stresses rise less than σ_zz), and it is not meant to be 1.
* **Noise**: the signal is `ΔP_pist ≈ 0.03` on a 1.5 baseline, so the normalised curves amplify the raw scatter ~50×.
  The empty bins beyond the pistons and the bins within `res_wall_margin` (3 σ) of a wet-piston plane are blanked; the
  load-piston-face and support-face bins are the known edge-bin artefacts and are left in.  `PARTIAL_NORM = 'raw'`
  draws the original figures only.
* **`FLIP_Z = True`**: every z-profile figure is drawn with the load piston / feed side on the left and the support /
  permeate side on the right, as in Marioni et al.  The x axis is inverted (`z/L`, `Z` and `ζ` label the same
  coordinates as before and decrease to the right); it is plotting only — no mask, window or fit sees it — and
  `FLIP_Z = False` restores the previous orientation pixel for pixel.

* **2026-10-07**: the normalised `P_th` figure (under 14) carries the guide `1 − (4/3)(G/M)` across the gel — the trace of
  the load when the lateral stresses carry `(M − 2G)/M` of the axial one (`G = G_REF` = 0.2, `M = M_net`; `M_REF`
  overrides).  Flipped axes now run 0 → 1 as `1 − z/L` and name their ends (load piston → support); σ- and ζ-unit axes
  stay inverted and say so in their labels.

* **2026-10-07 pm**: `G_COMP_REF` = 0.05 draws a second `1 − (4/3)(G/M)` guide (0.83 with `G_comp`, 0.32 with `G_shear`;
  measured 0.84).
